# 🚬 흡연 분류 V16 - LOCKED V4 + AutoML

## 핵심 원칙
- ✅ **V4 BASELINE LOCKED** - 재작성/변형 금지, 그대로 복사
- ✅ **AutoML은 옵션** - OOF에서 baseline을 이길 때만 채택
- ✅ **검증 통과 전 진행 금지** - raise로 강제

---

## CONFIG

In [ ]:
#===========================================
# CONFIG
#===========================================
USE_AUTOML = True  # AutoML 실행 여부
AUTOML_TIME_BUDGET = 120  # 초/fold

print("✅ CONFIG: USE_AUTOML =", USE_AUTOML)

## STEP 0: 환경 설정 + 안전장치

In [ ]:
!pip install -q xgboost lightgbm catboost flaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# 경로 설정 (고정)
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

os.makedirs(result_path, exist_ok=True)

In [ ]:
#===========================================
# 안전장치: 파일 존재 여부
#===========================================
print("=" * 60)
print("🔍 안전장치: 파일 존재 여부")
print("=" * 60)

files_check = {
    'train.csv': train_path,
    'test.csv': test_path,
    'sample_submission.csv': submission_path
}
all_exist = True
for name, path in files_check.items():
    exists = os.path.exists(path)
    print(f"   {name}: {'OK' if exists else 'NOT FOUND'}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("❌ 필수 파일이 없습니다.")
print("\n✅ 모든 파일 존재 확인")

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 데이터 로드 및 기본 검증
train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
submission_raw = pd.read_csv(submission_path)

print(f"\nShape: Train={train_raw.shape}, Test={test_raw.shape}")
print(f"\nTrain 컬럼: {train_raw.columns.tolist()}")
print(f"Test 컬럼: {test_raw.columns.tolist()}")

# 중복 컬럼
train_dup = train_raw.columns[train_raw.columns.duplicated()].tolist()
test_dup = test_raw.columns[test_raw.columns.duplicated()].tolist()
print(f"\n중복 컬럼: Train={train_dup if train_dup else '없음'}, Test={test_dup if test_dup else '없음'}")

# 결측치 TOP20
print("\n결측치 비율 (상위 20):")
missing = (train_raw.isnull().sum() / len(train_raw) * 100).sort_values(ascending=False)
print(missing.head(20).to_string())

---
# ═══════════════════════════════════════════════════════════
# STEP 1: V4 BASELINE (LOCKED) - 절대 수정 금지
# ═══════════════════════════════════════════════════════════
# 아래 코드는 smoking_classification_v4_final.ipynb에서 그대로 복사함

In [ ]:
#===========================================
# V4 BASELINE - LOCKED (수정 금지)
#===========================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

print("✅ 라이브러리 임포트 완료!")

In [ ]:
# V4 BASELINE - 데이터 로드
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"흡연자 비율: {train['label'].mean()*100:.2f}%")

# ID 처리
train_df = train.copy()
test_df = test.copy()

if 'ID' in test_df.columns:
    test_id = test_df['ID'].copy()
else:
    test_id = test_df['id'].copy() if 'id' in test_df.columns else None

train_df = train_df.drop(['ID', 'id'], axis=1, errors='ignore')
test_df = test_df.drop(['ID', 'id'], axis=1, errors='ignore')

X = train_df.drop('label', axis=1)
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

feature_cols = X.columns.tolist()
print(f"원본 특성 수: {len(feature_cols)}")

In [ ]:
# V4 BASELINE - 피처 엔지니어링
def create_features_v4(df):
    """V4 피처 엔지니어링"""
    df = df.copy()
    col_map = {c: c.lower() for c in df.columns}
    df_l = df.rename(columns=col_map)
    cols = df_l.columns.tolist()
    
    # 1. 콜레스테롤 관련
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df_l['hdl'] / (df_l['ldl'] + 1)
        df['LDL_HDL_ratio'] = df_l['ldl'] / (df_l['hdl'] + 1)
    if 'cholesterol' in cols and 'hdl' in cols:
        df['Atherogenic_idx'] = (df_l['cholesterol'] - df_l['hdl']) / (df_l['hdl'] + 1)
    if 'triglyceride' in cols and 'hdl' in cols:
        df['TG_HDL_ratio'] = df_l['triglyceride'] / (df_l['hdl'] + 1)
    
    # 2. 간 기능
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df_l['gtp'])
        df['GTP_sq'] = df_l['gtp'] ** 2
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df_l['ast'] / (df_l['alt'] + 1)
    
    # 3. 헤모글로빈
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df_l['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df_l['hemoglobin'])
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['Hemo_x_GTP'] = df_l['hemoglobin'] * df_l['gtp']
    
    # 4. 혈압
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df_l['systolic'] - df_l['diastolic']
        df['MAP'] = df_l['diastolic'] + (df_l['systolic'] - df_l['diastolic']) / 3
    
    # 5. 체형
    if 'height' in cols and 'weight' in cols:
        df['BMI_calc'] = df_l['weight'] / ((df_l['height']/100) ** 2 + 0.01)
    
    # 6. 시력
    eye_cols = [c for c in cols if 'eyesight' in c]
    if len(eye_cols) >= 2:
        df['Eyesight_avg'] = df_l[eye_cols].mean(axis=1)
    
    # 7. 나이 상호작용
    if 'age' in cols:
        age = df_l['age']
        df['Age_sq'] = age ** 2
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = age * df_l['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = age * df_l['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = age * df_l['triglyceride']
    
    # 8. 중성지방
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df_l['triglyceride'])
    
    # 9. 혈당
    fbs_cols = [c for c in cols if 'blood' in c or 'fasting' in c]
    if len(fbs_cols) > 0:
        df['FBS_log'] = np.log1p(df_l[fbs_cols[0]])
    
    # 10. 건강 통계
    health_cols = [c for c in cols if c in ['systolic','diastolic','hemoglobin','triglyceride','cholesterol','hdl','gtp']]
    if len(health_cols) >= 3:
        df['Health_mean'] = df_l[health_cols].mean(axis=1)
        df['Health_std'] = df_l[health_cols].std(axis=1)
    
    df = df.fillna(0).replace([np.inf, -np.inf], 0)
    return df

X_fe = create_features_v4(X)
X_test_fe = create_features_v4(X_test)
print(f"피처 엔지니어링 후: {X_fe.shape[1]}개")

In [ ]:
# V4 BASELINE - Feature Selection (상위 30개)
print("=" * 50)
print("🔍 Feature Selection: 중요 피처 선택")
print("=" * 50)

# 스케일링
scaler_temp = StandardScaler()
X_temp = scaler_temp.fit_transform(X_fe)

# LightGBM으로 피처 중요도 계산
lgb_temp = LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)
lgb_temp.fit(X_temp, y)

# 피처 중요도
importance = pd.DataFrame({
    'feature': X_fe.columns,
    'importance': lgb_temp.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top 20 중요 피처:")
print(importance.head(20).to_string(index=False))

# 상위 30개 피처 선택
TOP_N = 30
top_features = importance.head(TOP_N)['feature'].tolist()

X_selected = X_fe[top_features]
X_test_selected = X_test_fe[top_features]

print(f"\n✅ 선택된 피처 수: {len(top_features)}개")
print(f"선택된 피처: {top_features}")

In [ ]:
# V4 BASELINE - 최종 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)
X_test_scaled = scaler.transform(X_test_selected)

print(f"최종 Train: {X_scaled.shape}")
print(f"최종 Test: {X_test_scaled.shape}")

In [ ]:
# V4 BASELINE - K-Fold 교차 예측 (10 SEEDS)
print("=" * 50)
print("🎯 K-Fold 교차 예측 (데이터 100% 활용)")
print("=" * 50)

N_SPLITS = 5
SEEDS = [42, 123, 456, 789, 1004, 2024, 7777, 8888, 9999, 1234]

# OOF (Out-of-Fold) 예측 저장
oof_xgb = np.zeros(len(X_scaled))
oof_lgb = np.zeros(len(X_scaled))
oof_cat = np.zeros(len(X_scaled))
oof_rf = np.zeros(len(X_scaled))

# Test 예측 저장
test_xgb = np.zeros(len(X_test_scaled))
test_lgb = np.zeros(len(X_test_scaled))
test_cat = np.zeros(len(X_test_scaled))
test_rf = np.zeros(len(X_test_scaled))

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n--- Seed {seed} ({seed_idx+1}/{len(SEEDS)}) ---")
    
    kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_scaled, y)):
        X_tr, X_va = X_scaled[train_idx], X_scaled[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        # XGBoost
        xgb_m = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.02,
                              subsample=0.7, colsample_bytree=0.7,
                              random_state=seed, verbosity=0, use_label_encoder=False)
        xgb_m.fit(X_tr, y_tr)
        oof_xgb[val_idx] += xgb_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_xgb += xgb_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # LightGBM
        lgb_m = LGBMClassifier(n_estimators=500, max_depth=5, learning_rate=0.02,
                               subsample=0.7, colsample_bytree=0.7,
                               random_state=seed, verbose=-1)
        lgb_m.fit(X_tr, y_tr)
        oof_lgb[val_idx] += lgb_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_lgb += lgb_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # CatBoost
        cat_m = CatBoostClassifier(n_estimators=500, max_depth=5, learning_rate=0.02,
                                   random_state=seed, verbose=0)
        cat_m.fit(X_tr, y_tr)
        oof_cat[val_idx] += cat_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_cat += cat_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # Random Forest
        rf_m = RandomForestClassifier(n_estimators=300, max_depth=15,
                                      random_state=seed, n_jobs=-1)
        rf_m.fit(X_tr, y_tr)
        oof_rf[val_idx] += rf_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_rf += rf_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))

print("\n✅ K-Fold 교차 예측 완료!")

In [ ]:
# V4 BASELINE - 각 모델 OOF 성능 확인
print("\n📊 각 모델 OOF 성능:")
for name, oof in [('XGBoost', oof_xgb), ('LightGBM', oof_lgb), 
                   ('CatBoost', oof_cat), ('RF', oof_rf)]:
    pred = (oof >= 0.5).astype(int)
    acc = accuracy_score(y, pred)
    print(f"{name}: {acc:.5f}")

In [ ]:
# V4 BASELINE - Stacking 앙상블 (Meta 모델)
print("=" * 50)
print("🎯 Stacking 앙상블 (Meta 모델 학습)")
print("=" * 50)

# 1단계 모델 예측을 특성으로 결합
oof_stack = np.column_stack([oof_xgb, oof_lgb, oof_cat, oof_rf])
test_stack = np.column_stack([test_xgb, test_lgb, test_cat, test_rf])

print(f"Stacking 입력 shape: {oof_stack.shape}")

# 2단계: Meta 모델 (Logistic Regression)
meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_model.fit(oof_stack, y)

# Stacking 예측
oof_meta = meta_model.predict_proba(oof_stack)[:, 1]
test_meta = meta_model.predict_proba(test_stack)[:, 1]

print(f"\nMeta 모델 가중치: {meta_model.coef_[0]}")
print(f"(XGBoost, LightGBM, CatBoost, RF 순)")

In [ ]:
# V4 BASELINE - 단순 평균 vs Stacking 비교
oof_simple = (oof_xgb + oof_lgb + oof_cat + oof_rf) / 4

print("\n📊 앙상블 방법 비교:")
for name, oof in [('단순 평균', oof_simple), ('Stacking', oof_meta)]:
    pred = (oof >= 0.5).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    print(f"{name}: Acc={acc:.5f}, F1={f1:.5f}")

In [ ]:
# V4 BASELINE - 최적 임계값 탐색 (0.005 단위)
print("=" * 50)
print("🔍 최적 임계값 탐색 (0.005 단위 초세밀)")
print("=" * 50)

results_v4 = []

for thresh in np.arange(0.30, 0.60, 0.005):
    # 단순 평균
    pred_simple = (oof_simple >= thresh).astype(int)
    acc_simple = accuracy_score(y, pred_simple)
    
    # Stacking
    pred_stack = (oof_meta >= thresh).astype(int)
    acc_stack = accuracy_score(y, pred_stack)
    
    results_v4.append({
        'threshold': thresh,
        'acc_simple': acc_simple,
        'acc_stack': acc_stack,
        'best': max(acc_simple, acc_stack),
        'method': '단순평균' if acc_simple > acc_stack else 'Stacking'
    })

results_df_v4 = pd.DataFrame(results_v4)

print("\n📊 Threshold별 성능 (상위 15개):")
print(results_df_v4.nlargest(15, 'best')[['threshold', 'acc_simple', 'acc_stack', 'method']].to_string(index=False))

# 최적값 찾기
best_row_v4 = results_df_v4.loc[results_df_v4['best'].idxmax()]
baseline_threshold = best_row_v4['threshold']
baseline_method = best_row_v4['method']
baseline_acc = best_row_v4['best']

print(f"\n🏆 V4 BASELINE 최적 설정:")
print(f"   임계값: {baseline_threshold:.3f}")
print(f"   방법: {baseline_method}")
print(f"   OOF Accuracy: {baseline_acc:.5f}")

In [ ]:
# V4 BASELINE - 최적 방법 선택 및 baseline 제출 파일 생성
if baseline_method == 'Stacking':
    baseline_oof = oof_meta
    baseline_test = test_meta
else:
    baseline_oof = oof_simple
    baseline_test = (test_xgb + test_lgb + test_cat + test_rf) / 4

# Baseline 제출 파일 생성
baseline_pred = (baseline_test >= baseline_threshold).astype(int)
baseline_sub = submission.copy()
baseline_sub['label'] = baseline_pred
baseline_sub['label'] = baseline_sub['label'].astype(int)

baseline_filename = f'submission_v16_baseline_t{str(int(baseline_threshold*1000)).zfill(3)}.csv'
baseline_sub.to_csv(result_path + baseline_filename, index=False)

print(f"\n✅ V4 BASELINE 제출 파일 생성: {baseline_filename}")
print(f"   OOF Accuracy: {baseline_acc:.5f}")

---
# ═══════════════════════════════════════════════════════════
# BASELINE 검증 (통과 전 진행 금지)
# ═══════════════════════════════════════════════════════════

In [ ]:
#===========================================
# 필수 자동검증 - 통과 못하면 raise
#===========================================
print("=" * 60)
print("🔍 BASELINE 검증")
print("=" * 60)

errors = []

# 1. SEEDS 검증
if 'SEEDS' not in dir() or len(SEEDS) != 10:
    errors.append("SEEDS가 존재하지 않거나 len(SEEDS) != 10")
else:
    print(f"✅ SEEDS: {len(SEEDS)}개")

# 2. meta_model 검증
if 'meta_model' not in dir() or not isinstance(meta_model, LogisticRegression):
    errors.append("meta_model이 존재하지 않거나 LogisticRegression이 아님")
else:
    print(f"✅ meta_model: LogisticRegression")

# 3. top_features 검증
if 'top_features' not in dir() or len(top_features) == 0:
    errors.append("top_features가 존재하지 않거나 비어있음")
else:
    print(f"✅ top_features: {len(top_features)}개")

# 4. oof_stack (또는 oof_meta) 검증
if 'oof_meta' not in dir() or len(oof_meta) == 0:
    errors.append("oof_meta (stacking OOF)가 존재하지 않거나 비어있음")
else:
    print(f"✅ oof_meta: {len(oof_meta)}개")

# 5. baseline 제출 파일 검증
baseline_file_path = result_path + baseline_filename
if not os.path.exists(baseline_file_path):
    errors.append(f"baseline 제출 파일이 생성되지 않음: {baseline_filename}")
else:
    print(f"✅ baseline 제출 파일: {baseline_filename}")

# 검증 결과
if errors:
    print("\n❌ BASELINE 검증 실패:")
    for e in errors:
        print(f"   - {e}")
    raise ValueError("BASELINE 검증 실패! AutoML/제출 진행 불가.")
else:
    print("\n✅ BASELINE 검증 통과! AutoML 진행 가능.")
    BASELINE_VERIFIED = True

---
# ═══════════════════════════════════════════════════════════
# STEP 2: AutoML 후보 (옵션)
# ═══════════════════════════════════════════════════════════

In [ ]:
# 검증 통과 확인
if not BASELINE_VERIFIED:
    raise ValueError("BASELINE 검증이 통과되지 않았습니다.")

print("\n" + "=" * 60)
print("🚀 STEP 2: AutoML (FLAML)")
print("=" * 60)

automl_acc = 0
automl_threshold = 0.5
oof_automl = None
test_automl = None

if not USE_AUTOML:
    print("⏩ AutoML 비활성화 (USE_AUTOML=False)")
else:
    from flaml import AutoML
    
    # AutoML은 baseline과 동일한 입력 사용 (X_scaled)
    oof_automl = np.zeros(len(X_scaled))
    test_automl = np.zeros(len(X_test_scaled))
    
    cv_automl = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for fold, (tr_idx, va_idx) in enumerate(cv_automl.split(X_scaled, y)):
        print(f"\n--- AutoML Fold {fold+1}/5 ---")
        
        X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        automl = AutoML()
        automl.fit(
            X_tr, y_tr,
            task='classification', metric='accuracy',
            time_budget=AUTOML_TIME_BUDGET, verbose=0
        )
        
        oof_automl[va_idx] = automl.predict_proba(X_va)[:, 1]
        test_automl += automl.predict_proba(X_test_scaled)[:, 1] / 5
        
        fold_acc = accuracy_score(y_va, (oof_automl[va_idx] >= 0.5).astype(int))
        print(f"   Best: {automl.best_estimator}, Fold Acc: {fold_acc:.5f}")
    
    # AutoML threshold sweep
    best_automl_t, best_automl_acc = 0.5, 0
    for t in np.arange(0.30, 0.70, 0.01):
        acc = accuracy_score(y, (oof_automl >= t).astype(int))
        if acc > best_automl_acc:
            best_automl_acc = acc
            best_automl_t = round(t, 2)
    
    automl_acc = best_automl_acc
    automl_threshold = best_automl_t
    
    print(f"\n✅ AutoML 최종 OOF Accuracy: {automl_acc:.5f} (t={automl_threshold})")

---
# ═══════════════════════════════════════════════════════════
# STEP 3: 최종 채택 + 제출
# ═══════════════════════════════════════════════════════════

In [ ]:
print("\n" + "=" * 60)
print("📊 STEP 3: 최종 비교 및 채택")
print("=" * 60)

# OOF 비교표 생성
oof_summary = pd.DataFrame({
    'Method': ['BASELINE_V4_STACKING', 'AutoML_FLAML'],
    'Best_Threshold': [baseline_threshold, automl_threshold if USE_AUTOML else None],
    'Best_OOF_Acc': [baseline_acc, automl_acc if USE_AUTOML else 0]
})
print(oof_summary.to_string(index=False))

# 저장
oof_summary.to_csv(result_path + 'oof_summary_v16.csv', index=False)
print(f"\n✅ oof_summary_v16.csv 저장 완료")

# 채택 결정
if USE_AUTOML and automl_acc > baseline_acc:
    CHOSEN_METHOD = 'AutoML_FLAML'
    CHOSEN_OOF = oof_automl
    CHOSEN_TEST = test_automl
    CHOSEN_T = automl_threshold
    CHOSEN_ACC = automl_acc
    print(f"\n🏆 AutoML 채택 (OOF: {automl_acc:.5f} > {baseline_acc:.5f})")
else:
    CHOSEN_METHOD = 'BASELINE_V4_STACKING'
    CHOSEN_OOF = baseline_oof
    CHOSEN_TEST = baseline_test
    CHOSEN_T = baseline_threshold
    CHOSEN_ACC = baseline_acc
    print(f"\n🏆 BASELINE 채택 (OOF: {baseline_acc:.5f})")

In [ ]:
# 최종 제출 파일 생성 (3개)
print("\n" + "=" * 60)
print("📁 최종 제출 파일 생성")
print("=" * 60)

thresholds_final = [
    round(CHOSEN_T - 0.02, 3),
    round(CHOSEN_T, 3),
    round(CHOSEN_T + 0.02, 3)
]

method_short = 'baseline' if 'BASELINE' in CHOSEN_METHOD else 'automl'
saved_files = []

for t in thresholds_final:
    pred = (CHOSEN_TEST >= t).astype(int)
    oof_pred = (CHOSEN_OOF >= t).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    t_str = str(int(t * 1000)).zfill(3)
    if t == CHOSEN_T:
        fname = f'submission_v16_{method_short}_t{t_str}_best.csv'
    else:
        fname = f'submission_v16_{method_short}_t{t_str}.csv'
    
    sub.to_csv(result_path + fname, index=False)
    saved_files.append(fname)
    
    n1 = (pred == 1).sum()
    marker = "⭐" if t == CHOSEN_T else "  "
    print(f"{marker} {fname}: t={t:.3f}, OOF_Acc={oof_acc:.5f}, 흡연={n1}")

print(f"\n✅ {len(saved_files)}개 파일 저장 완료")

In [ ]:
# 최종 요약
print("\n" + "=" * 60)
print("🎉 V16 완료")
print("=" * 60)

print(f"\n🏆 최종: method={CHOSEN_METHOD} / OOF={CHOSEN_ACC:.5f} / best_t={CHOSEN_T}")

print(f"\n📁 저장된 파일:")
for f in saved_files:
    print(f"   ✅ {result_path}{f}")
print(f"   ✅ {result_path}oof_summary_v16.csv")

In [ ]:
from google.colab import files

best_fname = f'submission_v16_{method_short}_t{str(int(CHOSEN_T*1000)).zfill(3)}_best.csv'
files.download(result_path + best_fname)
print(f"\n📥 다운로드: {best_fname}")

In [ ]:
print("\n📥 추가 다운로드:")
for f in saved_files:
    if 'best' not in f:
        files.download(result_path + f)
        print(f"   ✅ {f}")